In [14]:
!pip -q install pandas numpy scikit-learn nltk tqdm transformers torch sentencepiece


In [15]:
import pandas as pd

base_path = "/content/"
reviews = pd.read_csv(base_path + "olist_order_reviews_dataset.csv")
order_items = pd.read_csv(base_path + "olist_order_items_dataset.csv")

reviews["review_comment_message"] = reviews["review_comment_message"].fillna("").astype(str)
reviews["has_text"] = reviews["review_comment_message"].str.strip().str.len() > 0

review_products = reviews.merge(
    order_items[["order_id", "product_id"]],
    on="order_id",
    how="inner"
)
review_products = review_products[review_products["has_text"]].copy()


In [16]:
top_products = (
    review_products.groupby("product_id")["review_comment_message"]
    .count()
    .sort_values(ascending=False)
    .head(500)
    .index
)

subset = review_products[review_products["product_id"].isin(top_products)].copy()

agg = (
    subset.groupby("product_id")["review_comment_message"]
    .apply(lambda x: " || ".join([t[:300] for t in x.tolist()[:20]]))
    .reset_index()
    .rename(columns={"review_comment_message": "reviews_blob"})
)


In [17]:
from transformers import pipeline
sent_pipe = pipeline("sentiment-analysis", model="cardiffnlp/twitter-xlm-roberta-base-sentiment", truncation=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Device set to use cpu


In [18]:
from tqdm import tqdm

def sentiment_label(score_idx):
    # cardiffnlp retorna labels: negative / neutral / positive
    return score_idx

sentiments = []
for _, r in tqdm(agg.iterrows(), total=len(agg)):
    text = r["reviews_blob"][:1000]  # limita
    out = sent_pipe(text)[0]
    sentiments.append({
        "product_id": r["product_id"],
        "sentiment_label": out["label"].lower(),
        "sentiment_confidence": out["score"]
    })

sent_df = pd.DataFrame(sentiments)
sent_df.head()


100%|██████████| 500/500 [06:30<00:00,  1.28it/s]


,product_id,sentiment_label,sentiment_confidence
0,00de7f393d962717eeeb2d7131a40dba,neutral,0.449786
1,014a8a503291921f7b004a5215bb3c36,neutral,0.392200
2,0152f69b6cf919bcdaf117aa8c43e5a2,neutral,0.398814
3,017692475c1c954ff597feda05131d73,neutral,0.449765
4,027cdd14a677a5834bc67a9789db5021,neutral,0.480720


In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

vectorizer = TfidfVectorizer(
    max_features=2000,
    ngram_range=(1,2),
    stop_words=None
)

X = vectorizer.fit_transform(agg["reviews_blob"])
feature_names = np.array(vectorizer.get_feature_names_out())

def top_keywords(row_idx, k=8):
    row = X[row_idx].toarray().ravel()
    top_idx = row.argsort()[-k:][::-1]
    return ", ".join(feature_names[top_idx])

agg["top_keywords"] = [top_keywords(i) for i in range(len(agg))]
agg[["product_id","top_keywords"]].head()


,product_id,top_keywords
0,00de7f393d962717eeeb2d7131a40dba,"na cor, cor, peço, cor preta, que me, branca, ..."
1,014a8a503291921f7b004a5215bb3c36,"cubo, meio, nao, aguardo, esse, os, muito caro..."
2,0152f69b6cf919bcdaf117aa8c43e5a2,"so, duas, eu paguei, entregaram apenas, compre..."
3,017692475c1c954ff597feda05131d73,"muito da, que, loja lannister, base, estou agu..."
4,027cdd14a677a5834bc67a9789db5021,"perfume, embalagem, caixa, na caixa, pra, fals..."


In [20]:
import re

delivery_kw = re.compile(r"\b(entrega|prazo|atras|transport|correio|chegou|frete)\w*", re.IGNORECASE)
quality_kw  = re.compile(r"\b(qualidade|defeit|quebr|ruim|bom|excelent|material|acabamento)\w*", re.IGNORECASE)
price_kw    = re.compile(r"\b(preço|caro|barato|custo|valor|promo)\w*", re.IGNORECASE)

def has_kw(pat, txt):
    return bool(pat.search(txt))

agg["delivery_mentions"] = agg["reviews_blob"].apply(lambda t: has_kw(delivery_kw, t))
agg["quality_mentions"]  = agg["reviews_blob"].apply(lambda t: has_kw(quality_kw, t))
agg["price_mentions"]    = agg["reviews_blob"].apply(lambda t: has_kw(price_kw, t))

agg[["product_id","delivery_mentions","quality_mentions","price_mentions"]].head()


,product_id,delivery_mentions,quality_mentions,price_mentions
0,00de7f393d962717eeeb2d7131a40dba,True,True,False
1,014a8a503291921f7b004a5215bb3c36,True,True,True
2,0152f69b6cf919bcdaf117aa8c43e5a2,True,True,True
3,017692475c1c954ff597feda05131d73,True,True,True
4,027cdd14a677a5834bc67a9789db5021,True,True,True


In [21]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [22]:
import json

def llm_json_features(text):
    prompt = f"""<|system|>
You are a data analyst. Output ONLY valid JSON.
<|user|>
Extract:
- main_topics (max 5)
- pain_points (max 5)
- suggested_improvements (max 5)
- summary (one sentence)
Text:
{text}

Return JSON with keys: main_topics, pain_points, suggested_improvements, summary
<|assistant|>
"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=220)
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)

    # tenta achar o trecho JSON dentro do texto
    start = decoded.find("{")
    end = decoded.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("No JSON found")
    return json.loads(decoded[start:end+1])


In [23]:
sample = agg.head(50).copy()

llm_rows, llm_errors = [], []
for _, r in tqdm(sample.iterrows(), total=len(sample)):
    try:
        js = llm_json_features(r["reviews_blob"][:1200])
        llm_rows.append({"product_id": r["product_id"], **js})
    except Exception as e:
        llm_errors.append({"product_id": r["product_id"], "error": str(e)})

llm_df = pd.DataFrame(llm_rows)
print("LLM ok:", len(llm_df), "errors:", len(llm_errors))
llm_df.head()

100%|██████████| 50/50 [1:12:58<00:00, 87.57s/it]

LLM ok: 27 errors: 23


,product_id,main_topics,pain_points,suggested_improvements,summary
0,00de7f393d962717eeeb2d7131a40dba,"[preta, branca]","[preta, branca]","[preta, branca]",O produto foi comprado na cor preta e enviado ...
1,0152f69b6cf919bcdaf117aa8c43e5a2,"[Tudo ok., Otimo, Empresa e produto de alta qu...",[Muito OBRIGADA!],"[O produto chegou antes do prazo, vale muito]","O tecido não é algodão, mas pelo preço vale su..."
2,017692475c1c954ff597feda05131d73,[5],[5],[5],5 out of 5 stars
3,044f05bc9de36e8a693a83e4bc79dd0d,"[Satisfeito, pain_points, suggested_improvemen...",[Satisfeito],[Produto de má qualidade],"Satisfeito || O produto foi para o lixo, não ..."
4,054515fd15bc1a2029f10de97ffa9120,"[recomendo, O produto chegou antes do prazo., ...","[Não recebi o produto e nem reembolso, Ainda n...",Agradeço por entregarem antes do prazo...,Bom produto


In [24]:
llm_errors[:5], len(llm_errors)

([{'product_id': '014a8a503291921f7b004a5215bb3c36', 'error': 'No JSON found'},
  {'product_id': '027cdd14a677a5834bc67a9789db5021', 'error': 'No JSON found'},
  {'product_id': '036734b5a58d5d4f46b0616ddc047ced', 'error': 'No JSON found'},
  {'product_id': '060c17562f97e5bb60bc0dfa4dd5b3f2', 'error': 'No JSON found'},
  {'product_id': '06edb72f1e0c64b14c5b79353f7abea3',
   'error': 'No JSON found'}],
 23)

In [25]:
failed_pids = [e["product_id"] for e in llm_errors]
len(failed_pids), failed_pids[:5]

(23,
 ['014a8a503291921f7b004a5215bb3c36',
  '027cdd14a677a5834bc67a9789db5021',
  '036734b5a58d5d4f46b0616ddc047ced',
  '060c17562f97e5bb60bc0dfa4dd5b3f2',
  '06edb72f1e0c64b14c5b79353f7abea3'])

In [26]:
import json
import torch

def llm_json_features_v2(text):
    prompt = f"""<|system|>
You are a data analyst. Output ONLY valid JSON.
Rules:
- main_topics, pain_points, suggested_improvements must be ARRAYS OF STRINGS.
- NEVER output numbers alone like [5].
- Keep each list with max 5 items.
- summary must be a short sentence (max 160 chars).
<|user|>
Extract features from the text (Portuguese allowed).
Text:
{text}

Return ONLY this JSON schema:
{{
  "main_topics": ["..."],
  "pain_points": ["..."],
  "suggested_improvements": ["..."],
  "summary": "..."
}}
<|assistant|>
"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=False,          # mais determinístico
            temperature=0.0,
            top_p=1.0
        )

    decoded = tokenizer.decode(out[0], skip_special_tokens=True)

    start = decoded.find("{")
    end = decoded.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("No JSON found")

    return json.loads(decoded[start:end+1])


In [27]:
from tqdm import tqdm
import pandas as pd

retry = agg[agg["product_id"].isin(failed_pids)].copy()

retry_rows, retry_errors = [], []

for _, r in tqdm(retry.iterrows(), total=len(retry)):
    try:
        js = llm_json_features_v2(r["reviews_blob"][:1200])
        retry_rows.append({"product_id": r["product_id"], **js})
    except Exception as e:
        retry_errors.append({"product_id": r["product_id"], "error": str(e)})

retry_df = pd.DataFrame(retry_rows)
print("retry ok:", len(retry_df), "retry errors:", len(retry_errors))
retry_df.head()


100%|██████████| 23/23 [26:04<00:00, 68.04s/it]

retry ok: 14 retry errors: 9


,product_id,main_topics,pain_points,suggested_improvements,summary
0,014a8a503291921f7b004a5215bb3c36,[...],[...],[...],...
1,027cdd14a677a5834bc67a9789db5021,[...],[...],[...],...
2,036734b5a58d5d4f46b0616ddc047ced,[...],[...],[...],...
3,060c17562f97e5bb60bc0dfa4dd5b3f2,[...],[...],[...],...
4,06edb72f1e0c64b14c5b79353f7abea3,[...],[...],[...],...


In [28]:
llm_df_all = pd.concat([llm_df, retry_df], ignore_index=True)
llm_df_all = llm_df_all.drop_duplicates(subset=["product_id"], keep="last")

print("LLM total ok:", len(llm_df_all))
llm_df_all.head()

LLM total ok: 41


,product_id,main_topics,pain_points,suggested_improvements,summary
0,00de7f393d962717eeeb2d7131a40dba,"[preta, branca]","[preta, branca]","[preta, branca]",O produto foi comprado na cor preta e enviado ...
1,0152f69b6cf919bcdaf117aa8c43e5a2,"[Tudo ok., Otimo, Empresa e produto de alta qu...",[Muito OBRIGADA!],"[O produto chegou antes do prazo, vale muito]","O tecido não é algodão, mas pelo preço vale su..."
2,017692475c1c954ff597feda05131d73,[5],[5],[5],5 out of 5 stars
3,044f05bc9de36e8a693a83e4bc79dd0d,"[Satisfeito, pain_points, suggested_improvemen...",[Satisfeito],[Produto de má qualidade],"Satisfeito || O produto foi para o lixo, não ..."
4,054515fd15bc1a2029f10de97ffa9120,"[recomendo, O produto chegou antes do prazo., ...","[Não recebi o produto e nem reembolso, Ainda n...",Agradeço por entregarem antes do prazo...,Bom produto


In [29]:
def clean_list(v, k=5):
    if v is None:
        return []
    if isinstance(v, str):
        v = [v]
    if not isinstance(v, list):
        return []
    out = []
    for item in v:
        if item is None:
            continue
        s = str(item).strip()
        if not s:
            continue
        # remove itens que são só número tipo "5"
        if s.isdigit():
            continue
        # remove coisas muito genéricas
        if s.lower() in {"satisfeito", "ok", "muito obrigado", "bom produto"}:
            continue
        out.append(s)
    # remove duplicados e limita
    uniq = []
    for s in out:
        if s not in uniq:
            uniq.append(s)
    return uniq[:k]

def clean_summary(s):
    if s is None:
        return ""
    s = str(s).replace("\n", " ").strip()
    return s[:160]

for col in ["main_topics", "pain_points", "suggested_improvements"]:
    if col in llm_df_all.columns:
        llm_df_all[col] = llm_df_all[col].apply(lambda x: clean_list(x, k=5))

if "summary" in llm_df_all.columns:
    llm_df_all["summary"] = llm_df_all["summary"].apply(clean_summary)

llm_df_all.head()


,product_id,main_topics,pain_points,suggested_improvements,summary
0,00de7f393d962717eeeb2d7131a40dba,"[preta, branca]","[preta, branca]","[preta, branca]",O produto foi comprado na cor preta e enviado ...
1,0152f69b6cf919bcdaf117aa8c43e5a2,"[Tudo ok., Otimo, Empresa e produto de alta qu...",[Muito OBRIGADA!],"[O produto chegou antes do prazo, vale muito]","O tecido não é algodão, mas pelo preço vale su..."
2,017692475c1c954ff597feda05131d73,[],[],[],5 out of 5 stars
3,044f05bc9de36e8a693a83e4bc79dd0d,"[pain_points, suggested_improvements, summary]",[],[Produto de má qualidade],"Satisfeito || O produto foi para o lixo, não ..."
4,054515fd15bc1a2029f10de97ffa9120,"[recomendo, O produto chegou antes do prazo., ...","[Não recebi o produto e nem reembolso, Ainda n...",[Agradeço por entregarem antes do prazo...],Bom produto


In [30]:
final_df = agg.merge(sent_df, on="product_id", how="left")

# se você já criou keywords e flags no agg, elas já estão aqui.
# se criou em outro DF, faça merge igual.

final_df = final_df.merge(llm_df_all, on="product_id", how="left")

print("final rows:", len(final_df), "unique products:", final_df["product_id"].nunique())
final_df.head()


final rows: 500 unique products: 500


,product_id,reviews_blob,top_keywords,delivery_mentions,quality_mentions,price_mentions,sentiment_label,sentiment_confidence,main_topics,pain_points,suggested_improvements,summary
0,00de7f393d962717eeeb2d7131a40dba,O produto foi comprado na cor preta e enviado ...,"na cor, cor, peço, cor preta, que me, branca, ...",True,True,False,neutral,0.449786,"[preta, branca]","[preta, branca]","[preta, branca]",O produto foi comprado na cor preta e enviado ...
1,014a8a503291921f7b004a5215bb3c36,aguardo a entrega do segundo produto. Até o pr...,"cubo, meio, nao, aguardo, esse, os, muito caro...",True,True,True,neutral,0.392200,[...],[...],[...],...
2,0152f69b6cf919bcdaf117aa8c43e5a2,Tudo ok. || Otimo || Empresa e produto de alta...,"so, duas, eu paguei, entregaram apenas, compre...",True,True,True,neutral,0.398814,"[Tudo ok., Otimo, Empresa e produto de alta qu...",[Muito OBRIGADA!],"[O produto chegou antes do prazo, vale muito]","O tecido não é algodão, mas pelo preço vale su..."
3,017692475c1c954ff597feda05131d73,Entrega rápida e produto conforme anunciado. |...,"muito da, que, loja lannister, base, estou agu...",True,True,True,neutral,0.449765,[],[],[],5 out of 5 stars
4,027cdd14a677a5834bc67a9789db5021,Não dei nota 5 pq não veio na mesma embalagem....,"perfume, embalagem, caixa, na caixa, pra, fals...",True,True,True,neutral,0.480720,[...],[...],[...],...


In [31]:
final_df.to_csv("/content/stg_product_text_features_free.csv", index=False)
